In [9]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [10]:
df=pd.read_pickle(r"D:\DataScience\GUVI\Local\Telangana_govt_Ration_distribution_clustering\master_dataset_cleaned.pkl")

In [11]:
#aggregate monthly data by shop

# Sort by shop and date
df = df.sort_values(["shopNo", "date"])

# Aggregate monthly data to shop level
shop_df = (
    df.groupby("shopNo")
      .agg(
          distCode=("distCode", "first"),
          distName=("distName_x", "first"),
          officeCode=("officeCode_x", "first"),
          officeName=("officeName_x", "first"),

          # Transaction features
          avg_transactions=("noOfTrans", "mean"),
          total_transactions=("noOfTrans", "sum"),
          max_transactions=("noOfTrans", "max"),
          min_transactions=("noOfTrans", "min"),
          transaction_std=("noOfTrans", "std"),

          # Ration-card features
          avg_cards=("noOfRcs", "mean"),
          max_cards=("noOfRcs", "max"),
          total_cards=("noOfRcs", "sum"),

          # Portability
          total_portability=("otherShopTransCnt", "sum"),
          avg_portability=("otherShopTransCnt", "mean"),
          
          # Commodity features
          avg_wheat=("wheat", "mean"),
          avg_sugar=("sugar", "mean"),
          avg_kerosene=("kerosene", "mean"),
          avg_salt=("salt", "mean"),

          #rice transaction for 3 cards
          avg_riceAfsc=("riceAfsc", "mean"),
          avg_riceFsc=("riceFsc", "mean"),
          avg_riceAap=("riceAap", "mean"),

          # Amount
          avg_amount=("totalAmount", "mean"),
          total_amount=("totalAmount", "sum")
      )
      .reset_index()
)

print(shop_df.shape)
shop_df.head()

(17491, 24)


,shopNo,distCode,distName,officeCode,officeName,avg_transactions,total_transactions,max_transactions,min_transactions,transaction_std,...,avg_portability,avg_wheat,avg_sugar,avg_kerosene,avg_salt,avg_riceAfsc,avg_riceFsc,avg_riceAap,avg_amount,total_amount
0,1407001,538,Mahbubnagar,538007,Koilkonda,265.344828,7695,294,202,20.243651,...,48.379310,0.0,0.000000,0.0,0.000000,1107.931034,5617.275862,0.000000,0.000000,0.0
1,1407002,538,Mahbubnagar,538007,Koilkonda,253.206897,7343,291,229,13.678396,...,48.793103,0.0,0.000000,0.0,0.000000,1322.758621,4690.206897,0.000000,0.000000,0.0
2,1407003,538,Mahbubnagar,538007,Koilkonda,423.827586,12291,433,416,3.713575,...,116.827586,0.0,0.000000,0.0,0.000000,1851.379310,8229.206897,19.655172,0.000000,0.0
3,1407004,538,Mahbubnagar,538007,Koilkonda,211.379310,6130,238,190,13.099656,...,7.413793,0.0,0.000000,0.0,0.275862,626.448276,4353.034483,5.862069,1.379310,40.0
4,1407005,538,Mahbubnagar,538007,Koilkonda,227.586207,6600,240,218,5.628481,...,10.000000,0.0,0.034483,0.0,0.000000,843.448276,4332.758621,0.000000,0.465517,13.5


utilization ratio helps identify unusually low or high beneficiary utilization

In [12]:
#utilization ratio
shop_df["utilization_ratio"] = (
    shop_df["total_transactions"] /
    shop_df["total_cards"].replace(0, np.nan)
)

portability_ratio identifies shops receiving unusually high numbers of beneficiaries from other shops

In [13]:
#probability ratio: 
shop_df["portability_ratio"] = (
    shop_df["total_portability"] /
    shop_df["total_transactions"].replace(0, np.nan)
)

transaction_cv finds shops with unusually unstable transaction volumes

In [14]:
#Transaction volatility
#Transaction Coefficient of Variation (CV): It measures how much a ration shop's transaction volume varies over time relative to its average
shop_df["transaction_cv"] = (
    shop_df["transaction_std"] /
    shop_df["avg_transactions"].replace(0, np.nan)
)

Proportion of the shop's average transactions involving AFSC, FSC, AAP

In [15]:
#share of each card
shop_df["afsc_transaction_share"] = (
    shop_df["avg_riceAfsc"] / shop_df["avg_transactions"]
)

shop_df["fsc_transaction_share"] = (
    shop_df["avg_riceFsc"] / shop_df["avg_transactions"]
)

shop_df["aap_transaction_share"] = (
    shop_df["avg_riceAap"] / shop_df["avg_transactions"]
)

In [16]:
pd.to_pickle(shop_df, r"D:\DataScience\GUVI\Local\Telangana_govt_Ration_distribution_clustering\feature_dataset.pkl")

In [17]:
shop_df.columns.to_list()

['shopNo',
 'distCode',
 'distName',
 'officeCode',
 'officeName',
 'avg_transactions',
 'total_transactions',
 'max_transactions',
 'min_transactions',
 'transaction_std',
 'avg_cards',
 'max_cards',
 'total_cards',
 'total_portability',
 'avg_portability',
 'avg_wheat',
 'avg_sugar',
 'avg_kerosene',
 'avg_salt',
 'avg_riceAfsc',
 'avg_riceFsc',
 'avg_riceAap',
 'avg_amount',
 'total_amount',
 'utilization_ratio',
 'portability_ratio',
 'transaction_cv',
 'afsc_transaction_share',
 'fsc_transaction_share',
 'aap_transaction_share']